# Allen-Cahn Phase Field — a Nonlinear Solve Inside Every Step

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/allen_cahn.ipynb)

Phase separation governed by

$$\partial_t c = \Delta c + \varepsilon^2 c\,(1 - c^2),$$

where random initial noise coarsens into sharp domains separated by thin
interfaces. Unlike the heat and wave notebooks, the operator here depends on
the current state, so **every time step runs its own Newton loop**: assemble
the tangent around $c$, solve for the correction, repeat until the residual
collapses.

The pattern — a `NodeAssembler` for the residual and an `ElementAssembler` for
its Jacobian — is the template for any nonlinear transient problem in
TensorMesh.

⏱️ *A few minutes on Colab's free CPU runtime: every step is a full Newton
solve. Lower `STEPS` for a quicker run.*

Docs: [Allen-Cahn Phase Field](https://docs.tensor-mesh.com/example_gallery/diffusion.html#allen-cahn-phase-field-allen-cahn-ac-py) · Source: [`examples/diffusion/allen-cahn/ac.py`](https://github.com/camlab-ethz/TensorMesh/blob/main/examples/diffusion/allen-cahn/ac.py)

In [ ]:
# Install TensorMesh (skipped automatically if it is already available, e.g. a local dev setup).
# The apt line provides the OpenGL utility library that gmsh -- TensorMesh's mesh generator --
# needs at import time; it is a no-op where the library is already present.
# ffmpeg backs matplotlib's movie writer, which turns the time series into mp4.
import importlib.util
if importlib.util.find_spec("tensormesh") is None:
    !apt-get -qq install -y libglu1-mesa ffmpeg > /dev/null 2>&1 || true
    %pip install -q tensormesh-fem==0.2.0

import contextlib
import os


@contextlib.contextmanager
def quiet():
    """Hide gmsh's meshing log, which is written below Python's stdout.
    Drop the ``with quiet():`` wrapper anywhere to see what the mesher is doing."""
    with open(os.devnull, "w") as null:
        saved = os.dup(1)
        os.dup2(null.fileno(), 1)
        try:
            yield
        finally:
            os.dup2(saved, 1)
            os.close(saved)

## Residual and tangent

Backward Euler in time. Writing the residual and its Jacobian as two small
assemblers keeps the Newton loop itself down to three lines.

In [ ]:
import torch

from tensormesh import Condenser, ElementAssembler, Mesh, NodeAssembler
from tensormesh.dataset import PoissonMultiFrequency

DT = 1e-6
EPSILON = 220


class Tangent(ElementAssembler):
    """Tangent (Jacobian) of the backward-Euler Allen-Cahn residual."""

    def __post_init__(self):
        self.dt = DT
        self.df = lambda x: -EPSILON ** 2 * (3 * x ** 2 - 1)

    def forward(self, u, v, gradu, gradv, c, gradc, cold):
        return -1.0 * ((1.0 / self.dt) * (u * v)
                       + (gradu @ gradv)
                       - self.df(c) * (u * v))


class Residual(NodeAssembler):
    """Backward-Euler residual  R = cdot v + grad(v).grad(c) - f(c) v."""

    def __post_init__(self):
        self.dt = DT
        self.f = lambda x: -EPSILON ** 2 * x * (x ** 2 - 1)

    def forward(self, v, gradv, c, gradc, cold):
        cdot = (c - cold) / self.dt
        return cdot * v + (gradv @ gradc) - self.f(c) * v

## Time stepping with an inner Newton loop

In [ ]:
STEPS = 60          # the full example runs 200
MAX_NEWTON = 50

with quiet():
    mesh = Mesh.gen_rectangle(chara_length=0.025, element_type="tri")
print(f"mesh: {mesh.n_points} nodes")

cold = PoissonMultiFrequency(K=24, r=1).source_term(mesh.points)
cs = [cold]

tangent = Tangent.from_mesh(mesh)
residual = Residual.from_mesh(mesh)

# Allen-Cahn is a closed system here: no Dirichlet data, so the "condenser" is
# an all-free mask and simply passes the system through.
condenser = Condenser(torch.zeros_like(mesh.boundary_mask, dtype=torch.bool))

for step in range(STEPS):
    c = cold
    for n_iter in range(MAX_NEWTON):
        point_data = {"c": c, "cold": cold}
        K_, R_ = condenser(tangent(mesh.points, point_data=point_data),
                           residual(mesh.points, point_data=point_data))
        c = c + condenser.recover(K_.solve(R_))
        rnorm = torch.linalg.norm(R_)
        if rnorm < 1e-10:
            break
    cold = c
    cs.append(cold)
    if step % 10 == 0 or step == STEPS - 1:
        print(f"step {step:3d}: {n_iter + 1:2d} Newton iterations, |R| = {rnorm:.2e}")

print(f"\n{len(cs)} frames; phi range [{cs[-1].min():.3f}, {cs[-1].max():.3f}]")

## Animation

Watch the interfaces sharpen and the small domains get eaten by their
neighbours — curvature-driven coarsening.

In [ ]:
mesh.plot(
    values={"phi": cs}, save_path="allen_cahn.mp4", dt=DT, show_mesh=False,
)
from IPython.display import Video
Video("allen_cahn.mp4", embed=True, width=780)

## Where to next

- [Convex-concave splitting](https://docs.tensor-mesh.com/example_gallery/diffusion.html) — an unconditionally stable variant of this same problem.
- [Hyperelastic beam](https://colab.research.google.com/github/camlab-ethz/TensorMesh/blob/main/notebooks/hyperelastic_beam.ipynb) — nonlinear solid mechanics by energy minimisation instead of Newton.